### RAGAS

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add pdfplumber

In [5]:
from langchain_community.document_loaders import PDFPlumberLoader
import pdfplumber

loader = PDFPlumberLoader("./data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]
len(docs)

19

In [6]:
docs[0].metadata

{'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': './data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [7]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [8]:
from ragas.testset import TestsetGenerator

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

generator_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=embeddings,
)

In [9]:
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors import NERExtractor
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer

transforms = [
    SummaryExtractor(llm=generator.llm),

    EmbeddingExtractor(
        embedding_model=generator.embedding_model,
        property_name="summary_embedding",
        embed_property_name="summary",
    ),
    NERExtractor(llm=generator.llm),
]

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm), 1.0),
]

In [10]:
testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=10,
    transforms=transforms,
    query_distribution=query_distribution,
    raise_exceptions=True,
)

Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000002729E3F1BC0> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-213' coro=<_async_in_context.<locals>.run_in_context() done, defined at d:\hanhwa0902\rag_two_branch\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-214' coro=<Kernel.shell_main() running at d:\hanhwa0902\rag_two_branch\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at d:\hanhwa0902\rag_two_branch\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:564]>
d:\hanhwa0902\rag_two_branch\.venv\Lib\site-packages\ragas\testset\synthesizers\single_hop\base.py:82

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [11]:
test_df = testset.to_pandas()

if test_df.empty:
    raise RuntiameError(
        "샘플이 0개입니다. 개체 추출 결과와 페르소나 매칭을 확인하세요."
    )

print(f"생성된 샘플 수: {len(test_df)}")

test_df

생성된 샘플 수: 10


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What are the key components of the executive o...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국, 안전하고...",The executive order signed by President Biden ...,AI Safety Researcher,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
1,G7가 히로시마 AI 프로세스를 통해 AI 기업을 위한 국제 행동강령을 만든 이유는...,"[SPRi AI Brief |\n2023-12월호\nG7, 히로시마 AI 프로세스를...",G7은 히로시마 AI 프로세스를 통해 AI 기업을 대상으로 AI 위험 식별과 완화를...,AI Governance Analyst,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,프랑스는 AI 안전성 정상회의에서 어떤 역할을 할 것인가요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n영국 AI 안전...,프랑스 정부는 1년 후 대면 정상회의를 개최할 예정입니다.,AI Safety Researcher,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,사라 앤더슨이 저작권 소송에서 어떤 역할을 했는지 설명해 주세요.,"[SPRi AI Brief |\n2023-12월호\n미국 법원, 예술가들이 생성 A...","사라 앤더슨은 2023년 1월에 미드저니, 스태빌리티AI, 디비언트아트를 상대로 저...",AI Governance Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer
4,FTC는 2023년 5월 31일에 어떤 조치를 취했나요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 연방거래위...,FTC는 2023년 5월 31일 동의를 받지 않고 어린이들의 음성과 위치 정보를 활...,AI Safety Researcher,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
5,What role does Aleph Alpha play in the context...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...","Aleph Alpha, a prominent AI company in Germany...",AI Governance Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
6,What are the main objectives of the AI 안전 기금 e...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n미국 프런티어 ...,"The AI 안전 기금, established by the Frontier Mode...",AI Governance Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
7,What is the role of MIT in the development of ...,"[SPRi AI Brief |\n2023-12월호\n코히어, 데이터 투명성 확보를 ...",MIT is one of the 12 institutions that collabo...,AI Safety Researcher,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
8,통이치엔원 2.0의 주요 성능 향상은 무엇인가요?,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,"통이치엔원 2.0은 복잡한 지침 이해, 광고문구 작성, 추론, 암기 등에서 성능이 ...",AI Safety Researcher,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
9,삼성 가우스 뭐야?,"[SPRi AI Brief |\n2023-12월호\n삼성전자, 자체 개발 생성 AI...","삼성 가우스는 삼성전자가 개발한 생성 AI 모델로, 언어, 코드, 이미지의 3개 모...",AI Governance Analyst,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer


In [12]:
test_df.to_csv("data/ragas_synthetic_dataset.csv", index=False, encoding="utf-8-sig")
print("저장완료")

저장완료
